# 🚀 TalentAI: Recruitment Intelligence Agent (Groq Version)
CrewAI + ChatGroq (llama-3.1-8b-instant)


In [ ]:
!pip install crewai gradio langchain-groq

In [ ]:
import os
from crewai import Agent, Task, Crew
import gradio as gr
from langchain_groq import ChatGroq


In [ ]:
# =========================
# GROQ API KEY SETUP
# =========================
if "GROQ_API_KEY" not in os.environ:
    from getpass import getpass
    os.environ["GROQ_API_KEY"] = getpass("Enter your GROQ API Key: ")

GROQ_API_KEY = os.getenv("GROQ_API_KEY")


In [ ]:
# =========================
# LLM CONFIG (Groq)
# =========================
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    api_key=GROQ_API_KEY
)


In [ ]:
# Agents
analyzer = Agent(role="Analyzer", goal="Evaluate resume vs JD", backstory="HR AI", llm=llm)
validator = Agent(role="Validator", goal="Validate output", backstory="Strict auditor", llm=llm)
formatter = Agent(role="Formatter", goal="Format report", backstory="Report generator", llm=llm)


In [ ]:
def run_talent_ai(jd, resume):
    t1 = Task(description=f"Compare JD and Resume:\n{jd}\n{resume}", agent=analyzer)
    r1 = Crew(agents=[analyzer], tasks=[t1]).kickoff()

    t2 = Task(description=f"Validate:\n{r1}", agent=validator)
    r2 = Crew(agents=[validator], tasks=[t2]).kickoff()

    t3 = Task(description=f"Format:\n{r2}", agent=formatter)
    r3 = Crew(agents=[formatter], tasks=[t3]).kickoff()

    return r3


In [ ]:
def ui(jd, resume):
    return run_talent_ai(jd, resume)

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.HTML("""
    <h1 style='text-align:center;'>🚀 TalentAI: Recruitment Intelligence Agent</h1>
    <div style='width:100%; overflow:hidden;'>
      <div style='animation: marquee 8s linear infinite;'>──────────────</div>
    </div>
    <style>
    @keyframes marquee {
      from { transform: translateX(0%); }
      to { transform: translateX(-100%); }
    }
    </style>
    """)

    jd = gr.Textbox(label="Job Description", lines=10)
    resume = gr.Textbox(label="Resume", lines=10)
    btn = gr.Button("Analyze Resume")
    out = gr.Markdown()

    btn.click(ui, [jd, resume], out)

demo.launch()
